# FinanceBench — 真实财报语料探索

逐 block 运行，理解 `src/policy_copilot/financebench.py`：把 8 份真实 10-K/10-Q 财报变成可检索的语料。

**如何运行：** VS Code 选内核 `.venv`，或终端 `uv run jupyter lab`。

> 注意：第一次运行 `build_corpus` 会下载 + 解析 PDF（约 1–2 分钟）；之后有缓存，秒开。

## 1. 加载 150 道题

In [ ]:
from policy_copilot.financebench import (
    build_corpus,
    load_rows,
    questions_for,
    select_documents,
)

rows = load_rows()
len(rows)

## 2. 看一道真实的题

每行有：`question`、`answer`、`doc_name`(来源文件)、`evidence`(黄金出处)。

In [ ]:
r = rows[0]
print("公司   :", r["company"])
print("来源   :", r["doc_name"])
print("问题   :", r["question"])
print("答案   :", r["answer"])
print("出处片段:", r["evidence"][0]["evidence_text"][:200], "...")

## 3. 选出精简子集（题最多的 8 份）

In [ ]:
doc_names = select_documents(rows, n=8)
doc_names

In [ ]:
questions = questions_for(rows, doc_names)
print(f"这 8 份文档覆盖 {len(questions)} 道题")

## 4. 构建语料（下载 + 解析，已缓存）

In [ ]:
corpus = build_corpus(rows, doc_names)
for f in corpus:
    print(f"{f.doc_id:30} {f.n_pages:>4} 页  {len(f.text):>9,} 字符")

## 5. “百页里捞针”——为什么检索是真问题

看一道题：它的黄金出处片段，在整篇文档里占多小的比例。

In [ ]:
doc = doc_names[0]
q = next(r for r in rows if r["doc_name"] == doc)
filing = next(f for f in corpus if f.doc_id == doc)
evidence = q["evidence"][0]["evidence_text"] if q["evidence"] else ""

print("问题:", q["question"])
print("答案:", q["answer"])
print(f"\n整篇文档 : {filing.n_pages} 页 / {len(filing.text):,} 字符")
print(f"黄金出处 : {len(evidence):,} 字符  (占全文 {len(evidence) / len(filing.text) * 100:.3f}%)")
print("\n出处预览:\n", evidence[:300])

## 6. 看一眼解析出的 Markdown

pymupdf4llm 把 PDF 转成 Markdown，标题层级保留下来，方便后面切块。

In [ ]:
print(filing.text[:1500])